# Guided Dataset Profiling & Comparative Quality Analysis

In real-world data pipelines, production data rarely arrives clean. This guided notebook demonstrates how to use **Prepix** to compare a clean, production-grade dataset against a realistic, flawed dataset.

Key learning objectives:
1. Identify silent schema flaws, missingness anomalies, and duplicated rows
2. Detect constant (zero-variance) columns and high-cardinality risks
3. Isolate distribution skewness and Tukey outliers
4. Use Prepix recommendations to systematically improve data hygiene


In [1]:
import pandas as pd
import numpy as np
import prepix as pp

pd.set_option('display.max_columns', 15)


## 1. Construct Clean Dataset vs Messy Dataset


In [2]:
# Clean reference dataset (10 unique employees, 0 missing, 0 duplicates)
df_clean = pd.DataFrame({
    "EmployeeID": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "FullName": ["Aman Verma", "Priya Sen", "Rahul Roy", "Sneha Rao", "Vikram Patel", "Neha Jain", "Amit Das", "Pooja Hegde", "Rohan Mehta", "Ananya Roy"],
    "Department": ["Engineering", "HR", "Sales", "Engineering", "Finance", "Marketing", "Engineering", "HR", "Sales", "Engineering"],
    "Salary": [75000.0, 62000.0, 54000.0, 82000.0, 95000.0, 58000.0, 88000.0, 60000.0, 52000.0, 78000.0],
    "PerformanceRating": [4.2, 3.8, 3.5, 4.7, 4.0, 3.9, 4.8, 3.6, 3.2, 4.5],
    "IsActive": [True, True, True, True, True, True, True, True, True, True]
})

# Messy dataset (injected duplicate rows, null values, constant column, extreme outlier)
df_messy = pd.DataFrame({
    "EmployeeID": [101, 102, 103, 104, 105, 105, 107, 108, 109, 110],  # Duplicate 105
    "FullName": ["Aman Verma", "Priya Sen", "Rahul Roy", "Sneha Rao", "Vikram Patel", "Vikram Patel", "Amit Das", None, "Rohan Mehta", "Ananya Roy"],
    "Department": ["Engineering", "HR", "Sales", "Engineering", "Finance", "Finance", "Engineering", "HR", "Sales", "Engineering"],
    "Salary": [75000.0, 62000.0, 54000.0, 82000.0, 95000.0, 95000.0, 88000.0, 60000.0, 52000.0, 999999.0],  # 999999 outlier!
    "PerformanceRating": [4.2, 3.8, None, 4.7, 4.0, 4.0, None, 3.6, 3.2, 4.5],  # Missing ratings
    "IsActive": [True, True, True, True, True, True, True, True, True, True],
    "CompanyStatus": ["Active"] * 10  # Constant column!
})

print(f"Clean shape: {df_clean.shape}, Messy shape: {df_messy.shape}")


Clean shape: (10, 6), Messy shape: (10, 7)


## 2. Generate Unified Prepix Reports


In [3]:
rep_clean = pp.report(df_clean)
rep_messy = pp.report(df_messy)

print('=== CLEAN DATASET REPORT ===')
print(rep_clean)


=== CLEAN DATASET REPORT ===
                    PREPIX QUALITY REPORT                    

1. DATASET OVERVIEW
--------------------------------------------------------------
  - Rows                        : 10
  - Columns                     : 6
  - Total Cells                 : 60
  - Memory Usage                : 1.65 KB
  - Numeric Columns             : 3
  - Categorical Columns         : 2
  - Datetime Columns            : 0
  - Boolean Columns             : 1
  - Other Columns               : 0
  - Total Missing Cells         : 0 (0.0%)
  - Columns with Missing        : 0
  - Duplicate Rows              : 0 (0.0%)

2. KEY FINDINGS & ISSUES
--------------------------------------------------------------
  [LOW   ] Column 'EmployeeID': Potential Identifier - 100% unique values (10/10)
  [LOW   ] Column 'FullName': Potential Identifier - 100% unique values (10/10)
  [LOW   ] Column 'IsActive': Zero Variance (Constant) - Contains single constant value across all 10 rows

3. RECOMMEND

### Auditing the Messy Dataset Report
Notice how Prepix automatically flags the duplicate row, the missing ratings, the constant column, and recommends remediation actions:


In [4]:
print('=== MESSY DATASET REPORT ===')
print(rep_messy)


=== MESSY DATASET REPORT ===
                    PREPIX QUALITY REPORT                    

1. DATASET OVERVIEW
--------------------------------------------------------------
  - Rows                        : 10
  - Columns                     : 7
  - Total Cells                 : 70
  - Memory Usage                : 2.24 KB
  - Numeric Columns             : 3
  - Categorical Columns         : 3
  - Datetime Columns            : 0
  - Boolean Columns             : 1
  - Other Columns               : 0
  - Total Missing Cells         : 3 (4.29%)
  - Columns with Missing        : 2
  - Duplicate Rows              : 1 (10.0%)

2. KEY FINDINGS & ISSUES
--------------------------------------------------------------
  [MEDIUM] Dataset: Duplicate Records - 1 duplicate rows (10.0%) detected
  [MEDIUM] Column 'PerformanceRating': Moderate Missingness - 2 missing values (20.0%)
  [LOW   ] Column 'IsActive': Zero Variance (Constant) - Contains single constant value across all 10 rows
  [LOW   ] C

## 3. Investigating the Prioritized Issues Table (`report.issues`)
Prepix assigns a severity rating (`HIGH`, `MEDIUM`, `LOW`) to each finding:


In [5]:
rep_messy.issues


,Severity,Column,Issue,Detail,Recommendation
0,Medium,Dataset,Duplicate Records,1 duplicate rows (10.0%) detected,Review and consider deduplicating using df.dro...
1,Medium,PerformanceRating,Moderate Missingness,2 missing values (20.0%),Consider median/mode imputation or indicator f...
2,Low,IsActive,Zero Variance (Constant),Contains single constant value across all 10 rows,Consider dropping constant column 'IsActive' p...
3,Low,CompanyStatus,Zero Variance (Constant),Contains single constant value across all 10 rows,Consider dropping constant column 'CompanyStat...
4,Low,Salary,Potential Outliers,1 values outside 1.5x IQR boundaries,Inspect distribution and consider robust scali...
5,Low,Salary,Highly Skewed Distribution,Skewness of 3.14 (right/positive-skewed),Consider log or power transformation on 'Salar...


## 4. Numerical Distribution & Outlier Detection
Examining the salary outlier (`999,999.0`) with `numeric_summary(extended=True)`:


In [6]:
num_clean = pp.numeric_summary(df_clean, extended=True)
num_messy = pp.numeric_summary(df_messy, extended=True)

print('Clean Salary Mean vs Messy Salary Mean:')
print('Clean:', num_clean[num_clean['Column'] == 'Salary'][['Mean', 'Median', 'Std', 'Outliers']].to_dict('records'))
print('Messy:', num_messy[num_messy['Column'] == 'Salary'][['Mean', 'Median', 'Std', 'Outliers']].to_dict('records'))

num_messy


Clean Salary Mean vs Messy Salary Mean:
Clean: [{'Mean': 70400.0, 'Median': 68500.0, 'Std': 15159.888595310396, 'Outliers': 0}]
Messy: [{'Mean': 166299.9, 'Median': 78500.0, 'Std': 293380.381189279, 'Outliers': 1}]


,Column,Data Type,Count,Missing Values,Mean,Median,Std,...,Range,IQR,Skewness,Kurtosis,Zeros,Negatives,Outliers
0,EmployeeID,int64,10,0,105.4,105.0,3.025815,...,9.0,4.500,0.122730,-1.179756,0,0,0
1,Salary,float64,10,0,166299.9,78500.0,293380.381189,...,947999.0,32750.000,3.144190,9.916145,0,0,1
2,PerformanceRating,float64,8,2,4.0,4.0,0.481070,...,1.5,0.525,-0.184773,-0.156516,0,0,0


## 5. Schema Contract Enforcement
We validate both datasets against our production data contract:


In [7]:
contract_schema = {
    "EmployeeID": {"type": "integer", "nullable": False, "unique": True},
    "FullName": {"type": "string", "nullable": False},
    "Department": "string",
    "Salary": {"type": "numeric", "nullable": False},
    "PerformanceRating": {"type": "float", "nullable": False},
}

print('Clean Dataset Validation:')
res_clean = pp.schema_report(df_clean, schema=contract_schema, allow_unexpected=False)
display(res_clean[res_clean['Status'] == 'FAIL'])

print('\nMessy Dataset Validation (Flagged Violations):')
res_messy = pp.schema_report(df_messy, schema=contract_schema, allow_unexpected=False)
display(res_messy[res_messy['Status'] == 'FAIL'])


Clean Dataset Validation:


name 'display' is not defined